<a href="https://colab.research.google.com/github/rezumistri2026/AI_LAB_ASSIGNMENTS/blob/main/AI_LAB_EXP_15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import warnings
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.layers import (
    Activation,
    BatchNormalization,
    Concatenate,
    Conv2D,
    Dense,
    Dropout,
    GlobalAveragePooling2D,
    GlobalMaxPooling2D,
    Input,
    MaxPooling2D,
)
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical

warnings.filterwarnings("ignore")
tf.random.set_seed(42)

# 1. Load & Preprocess Fashion-MNIST Data
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()
class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]
num_classes = len(class_names)

X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0
X_train = np.expand_dims(X_train, axis=-1)
X_test = np.expand_dims(X_test, axis=-1)

y_train_enc = to_categorical(y_train, num_classes)
y_test_enc = to_categorical(y_test, num_classes)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train_enc, test_size=0.15, random_state=42, stratify=y_train
)

# 2. Build 2D CNN Architecture with Dual Global Pooling
inputs = Input(shape=(28, 28, 1))
x = Conv2D(32, (3, 3), padding="same", use_bias=False)(inputs)
x = BatchNormalization()(x)
x = Activation("relu")(x)
x = MaxPooling2D((2, 2))(x)

x = Conv2D(64, (3, 3), padding="same", use_bias=False)(x)
x = BatchNormalization()(x)
x = Activation("relu")(x)
x = MaxPooling2D((2, 2))(x)

x = Conv2D(128, (3, 3), padding="same", use_bias=False)(x)
x = BatchNormalization()(x)
x = Activation("relu")(x)

max_p = GlobalMaxPooling2D()(x)
avg_p = GlobalAveragePooling2D()(x)
x = Concatenate()([max_p, avg_p])

x = Dense(128, activation="relu")(x)
x = Dropout(0.3)(x)
outputs = Dense(num_classes, activation="softmax")(x)

model = Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

# 3. Train Model
history = model.fit(
    X_tr, y_tr,
    epochs=20,
    batch_size=64,
    validation_data=(X_val, y_val),
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
        ModelCheckpoint("best_2d_cnn.keras", save_best_only=True, verbose=0)
    ],
    verbose=1,
)

# 4. Evaluate & Predict
y_prob = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

print(f"\nTest Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=class_names))

# 5. Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(9, 7))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=class_names, yticklabels=class_names, linewidths=0.5
)
plt.title("2D CNN Confusion Matrix (Fashion-MNIST)")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()